# Extending problem to search based on the graph alignment
Plan for the search and ranking problem:
1. Index all premises and arguments
2. Semantic candidate retrieval
3. Recovering local graph context
4. Run the alignment only on candidates
5. Combine semantic and structural relevance
6. Return multiple results level
7. Aggregate duplicate hits
8. Define what relevance means
9. Compare search systems

In [21]:
import pandas as pd
import numpy as np
import torch

import logging
from pathlib import Path

from cuda.bindings.cyruntime_types import cudaErrorLaunchFileScopedTex

from legal_data_doc_parts.argument_graph import (
    locate_project_resource,
    load_and_validate_documents,
    TransformerMeanPoolEmbedder,
    embed_node_attributes,
)

Env configuration

In [19]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("modular-argument-graph-search")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 64

## 1 - Index all premises and arguments

In [14]:
paragraph_dataset_mini = [
    "116062204", "116075685", "116080900", "116084352", "116084886",
    "116099338", "116102314", "116105977", "116106432", "116109598",
    "116110555", "116116081", "116118921", "116120654", "116121714",
    "116125191", "116126076", "116127831", "116141779", "116142769",
    "116149190", "116150283", "116167789", "116168634", "116170057",
    "116170822", "116192026", "116194637", "116196887", "116197578",
    "116199866", "116204676", "116213690", "116214232", "116220471",
    "116221948", "116226431", "116229939", "116232414", "116233794",
    "116245112", "116246991", "116248468", "116252607", "116253514",
    "116256069", "116257159", "116259036", "116260246", "116261015",
    "116261263", "116270112", "116277632", "116278949", "116283678",
    "116285648", "116289180", "116308208", "116310154", "116320626",
    "116321603", "116336690", "116339406", "116344031", "116350541",
    "116350825", "116358099", "116359089", "116376162", "116377931",
    "116385451", "116398308", "116410200", "116410462", "116414218",
    "116432495", "116434259", "116442566", "116462199", "116588653",
    "116681882", "116764649", "116780853", "116790074", "116794466",
    "116972204", "117069850", "117125134", "117130242", "117230819",
    "117251852", "117420480", "117440182", "117456818", "117460145",
    "117464049", "117489745", "117516936", "117556435",
]

assert paragraph_dataset_mini
assert len(paragraph_dataset_mini) == len(set(paragraph_dataset_mini))
logger.info("Configured %d documents", len(paragraph_dataset_mini))

[20/Sep/2026 04:04:35] INFO - Configured 99 documents


In [15]:
DATA_DIR = locate_project_resource(Path("document_data_parsing"))
ENCODER_DIR = locate_project_resource(
    Path("bge_model/bge-small-uk-law-v1_1/final/encoder")
)
TOKENIZER_DIR = locate_project_resource(
    Path("bge_model/bge-small-uk-law-v1_1/final/tokenizer")
)

dataset = load_and_validate_documents(paragraph_dataset_mini, DATA_DIR)
doc_args = dataset.documents
arguments_df = dataset.arguments
relations_df = dataset.relations

dataset_summary = pd.Series(
    {
        "documents": len(doc_args),
        "arguments": len(arguments_df),
        "relations": len(relations_df),
        "premises": int((arguments_df["argument_type"] == "PREMISE").sum()),
        "claims": int((arguments_df["argument_type"] == "CLAIM").sum()),
        "duplicate_source_target_pairs": int(
            relations_df.duplicated(["document_id", "source", "target"]).sum()
        ),
    },
    name="count",
)
display(dataset_summary.to_frame())

,count
documents,99
arguments,3941
relations,2928
premises,2710
claims,1231
duplicate_source_target_pairs,15


In [22]:
embeddings_model = TransformerMeanPoolEmbedder.from_pretrained(
    ENCODER_DIR,
    TOKENIZER_DIR,
    device=DEVICE,
    batch_size=BATCH_SIZE,
)

embedded_nodes = embed_node_attributes(arguments_df, embeddings_model, device=DEVICE)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

In [26]:
embedded_nodes.text_embeddings.shape

torch.Size([3941, 384])

## 2 - Semantic candidate retrieval

Given the `query` value(premise or argument) and `k` cut value we need to fetch all semantically-similar values

In [28]:
import torch.nn.functional as F

In [53]:
def query_semantic_values(query, k, from_embedding_nodes):
    query_embedding = embeddings_model.encode([query])

    print(f"queries for comparison: {query_embedding.shape}, for embeddings list: {from_embedding_nodes.shape}")

    topk_sim = F.cosine_similarity(from_embedding_nodes, query_embedding, dim=1).topk(k, largest=True)
    return topk_sim

In [69]:
query_1 = "містить тетрагідроканнабінол і є особливо небезпечним наркотичним засобом - канабісом"
topk_similar_matches_1 = query_semantic_values(query_1, 20, embedded_nodes.text_embeddings)
similar_df_1 = arguments_df.iloc[topk_similar_matches_1.indices]

queries for comparison: torch.Size([1, 384]), for embeddings list: torch.Size([3941, 384])


## 3 - Load global graph context

In [72]:
clustered_arguments = pd.read_parquet("clustered_arguments.parquet.gzip")
clustered_arguments.head()

,cluster_id,cluster_size,is_center,conductance,global_node_index,graph_index,local_node_index,document_id,argument_id,node_id,argument_type,argumentative_role,section,text
0,0,12,True,0.741757,1615,41,9,116246991,P08,116246991.P08,PREMISE,BASE_PREMISE,Встановлені судом обставини,"Окрім цього, в період часу з 15:35 до 15:43 го..."
1,0,12,False,0.741757,7,0,7,116062204,P24,116062204.P24,PREMISE,BASE_PREMISE,Встановлені судом обставини,"Так, ОСОБА_5 , за попередньою змовою та спільн..."
2,0,12,False,0.741757,327,5,0,116099338,P01,116099338.P01,PREMISE,BASE_PREMISE,Встановлені судом обставини,"ОСОБА_3 , 15.07.2023, приблизно о 20:00 год, п..."
3,0,12,False,0.741757,443,8,1,116106432,P13,116106432.P13,PREMISE,BASE_PREMISE,Встановлені судом обставини,"ОСОБА_4 пред`явлено обвинувачення в тому, що в..."
4,0,12,False,0.741757,652,13,5,116120654,P06,116120654.P06,PREMISE,BASE_PREMISE,Встановлені судом обставини,"Реалізуючи свій злочинний умисел ОСОБА_4 , дію..."


In [73]:
clustered_arguments.shape

(3941, 14)

Searching for the elements

In [79]:
similar_df_1["node_id"] = similar_df_1["document_id"].astype(str) + "." + similar_df_1["argument_id"].astype(str)
topk_similar = similar_df_1["node_id"].tolist()

In [87]:
filtered_values_df = clustered_arguments[clustered_arguments["node_id"].isin(topk_similar)]
filtered_values_df.groupby(by=["cluster_id"]).count()

,cluster_size,is_center,conductance,global_node_index,graph_index,local_node_index,document_id,argument_id,node_id,argument_type,argumentative_role,section,text
cluster_id,,,,,,,,,,,,,
80,1,1,1,1,1,1,1,1,1,1,1,1,1
84,2,2,2,2,2,2,2,2,2,2,2,2,2
237,1,1,1,1,1,1,1,1,1,1,1,1,1
256,3,3,3,3,3,3,3,3,3,3,3,3,3
259,3,3,3,3,3,3,3,3,3,3,3,3,3
270,2,2,2,2,2,2,2,2,2,2,2,2,2
383,1,1,1,1,1,1,1,1,1,1,1,1,1
391,1,1,1,1,1,1,1,1,1,1,1,1,1
443,1,1,1,1,1,1,1,1,1,1,1,1,1


In [94]:
clustered_arguments[clustered_arguments["cluster_id"] == 80]

,cluster_id,cluster_size,is_center,conductance,global_node_index,graph_index,local_node_index,document_id,argument_id,node_id,argument_type,argumentative_role,section,text
1089,80,13,True,0.3832,2648,67,7,116359089,P09,116359089.P09,PREMISE,BASE_PREMISE,Нормативні підстави,Згідно наказу Міністерства охорони здоров`я ві...
1090,80,13,False,0.3832,444,8,2,116106432,P14,116106432.P14,PREMISE,BASE_PREMISE,Докази та обставини вилучення,"Цього ж таки дня, близько 17:00 год., у м.Мост..."
1091,80,13,False,0.3832,683,14,0,116121714,P01,116121714.P01,PREMISE,BASE_PREMISE,Обставини кримінального правопорушення,"ОСОБА_3 , приблизно у грудні 2023 року, точног..."
1092,80,13,False,0.3832,746,16,0,116126076,P01,116126076.P01,PREMISE,BASE_PREMISE,Встановлені судом обставини,ОСОБА_2 маючи умисел на незаконне придбання та...
1093,80,13,False,0.3832,860,20,3,116149190,P04,116149190.P04,PREMISE,BASE_PREMISE,Встановлені судом обставини,який згідно таблиці І списку №1 «Переліку нарк...
1094,80,13,False,0.3832,1257,32,5,116213690,P06,116213690.P06,PREMISE,BASE_PREMISE,В С Т А Н О В И В:,"Впевнившись в тому, що у виявленому ним паперо..."
1095,80,13,False,0.3832,1695,42,36,116248468,P01,116248468.P01,PREMISE,BASE_PREMISE,Нормативні підстави,Відповідно до статті 12 Закону України «Про на...
1096,80,13,False,0.3832,1969,48,10,116260246,P12,116260246.P12,PREMISE,BASE_PREMISE,Нормативна кваліфікація речовини,PVP - згідно зі Списком №2 «Особливо небезпечн...
1097,80,13,False,0.3832,2408,59,0,116320626,P01,116320626.P01,PREMISE,BASE_PREMISE,Встановлені обставини,"передбачено, що обіг наркотичних засобів, псих..."
1098,80,13,False,0.3832,2516,63,9,116344031,P10,116344031.P10,PREMISE,BASE_PREMISE,Нормативні положення (наказ МОЗ),Згідно наказу МОЗ України № 188 від 01.08.2000...


In [96]:
clustered_arguments[clustered_arguments["document_id"] == "116359089"]

,cluster_id,cluster_size,is_center,conductance,global_node_index,graph_index,local_node_index,document_id,argument_id,node_id,argument_type,argumentative_role,section,text
133,8,19,True,0.071901,2652,67,11,116359089,P13,116359089.P13,PREMISE,BASE_PREMISE,Мотиви призначення покарання,"Згідно ст. 66 КК України, обставиною, що пом`я..."
188,11,10,False,0.449933,2660,67,19,116359089,P19,116359089.P19,PREMISE,BASE_PREMISE,Судові витрати,відповідно до вимог ч.2 ст. 124 КПК України
230,13,38,False,0.209471,2663,67,22,116359089,P22,116359089.P22,PREMISE,BASE_PREMISE,Процесуальні норми (керуючись),"На підставі викладеного, керуючись ст.ст. 100,..."
295,15,45,False,0.326270,2668,67,27,116359089,C03,116359089.C03,CLAIM,INTERMEDIATE_CONCLUSION,Мотиви призначення покарання,"На підставі викладеного, суд приходить до висн..."
421,24,14,False,0.046998,2653,67,12,116359089,P14,116359089.P14,PREMISE,BASE_PREMISE,Мотиви призначення покарання,"Згідно ст. 67 КК України, обставин, що обтяжую..."
433,25,18,False,0.022936,2672,67,31,116359089,C07,116359089.C07,CLAIM,FINAL_CONCLUSION,Речові докази,Питання про речові докази необхідно вирішити в...
484,30,10,False,0.360139,2662,67,21,116359089,P21,116359089.P21,PREMISE,BASE_PREMISE,Речові докази,відповідно до вимог ст. 100 КПК України
680,48,15,False,0.257225,2650,67,9,116359089,P11,116359089.P11,PREMISE,BASE_PREMISE,Процесуальні підстави,відповідно до ч.2 ст.382 КПК України
708,51,12,False,0.442230,2651,67,10,116359089,P12,116359089.P12,PREMISE,BASE_PREMISE,Мотиви призначення покарання,у відповідності до ст. 65 КК України
720,52,22,False,0.260869,2666,67,25,116359089,C01,116359089.C01,CLAIM,INTERMEDIATE_CONCLUSION,Правова кваліфікація,Кваліфікація дій ОСОБА_3 за ч. 1 ст. 309 КК Ук...


Query 2

In [97]:
query_2_1 = "містить тетрагідроканнабінол і є особливо небезпечним наркотичним засобом - канабісом"
query_2_2 = "щире каяття"

topk_similar_matches_2_1 = query_semantic_values(query_2_1, 10, embedded_nodes.text_embeddings)
topk_similar_matches_2_2 = query_semantic_values(query_2_2, 10, embedded_nodes.text_embeddings)

similar_df_2_1 = arguments_df.iloc[topk_similar_matches_2_1.indices]
similar_df_2_2 = arguments_df.iloc[topk_similar_matches_2_2.indices]

queries for comparison: torch.Size([1, 384]), for embeddings list: torch.Size([3941, 384])
queries for comparison: torch.Size([1, 384]), for embeddings list: torch.Size([3941, 384])


In [99]:
similar_df_2_2.head()

,global_node_index,graph_index,local_node_index,document_id,argument_id,node_id,argument_type,argumentative_role,section,text
853,853,19,21,116142769,C07,116142769.C07,CLAIM,INTERMEDIATE_CONCLUSION,Пом'якшуючі обставини,щире каяття
2504,2504,62,15,116339406,P12,116339406.P12,PREMISE,BASE_PREMISE,"Обставини, що пом`якшують покарання",щире каяття
1201,1201,30,19,116199866,P15,116199866.P15,PREMISE,BASE_PREMISE,"Обставини, що пом'якшують та обтяжують покарання",його щире каяття
1401,1401,36,13,116226431,P14,116226431.P14,PREMISE,BASE_PREMISE,Позиція обвинуваченого,Висловив щире каяття з приводу вчиненого.
3324,3324,82,48,116780853,P32,116780853.P32,PREMISE,BASE_PREMISE,Пояснення обвинуваченого,Щиро розкаявся.
